<a href="https://colab.research.google.com/github/sinbbang/L-IntelligentSystem/blob/main/%EC%A7%80%EB%8A%A5%EC%8B%9C%EC%8A%A4%ED%85%9C_10-1_ReAct_%EB%8C%80%ED%95%99%EC%83%9D%ED%99%9C_%EB%8F%84%EC%9A%B0%EB%AF%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 실습 1. ReAct 기반 대학생활 도우미

### 학습 목표
1. **Thought(판단) → Action(도구 호출) → Observation(실제 반환값 확인)** 흐름을 구현한다.
2. JSON 파일에서 **과제 마감일**을 조회하고 **남은 일수**를 계산한다.
3. 반복 실행 기록(Trace)을 읽고 **도구의 결과와 모델의 추측을 구분**한다.

> **중요:** 이 실습은 LLM API 없이 작동하는 **규칙 기반 ReAct 학습용 시뮬레이터**입니다. 자연어를 분석해 다음 행동을 선택하는 부분은 간단한 Python 규칙으로 구현합니다. 실제 LLM 기반 ReAct에서는 이 선택을 언어모델이 수행하고, 프로그램이 도구를 실행합니다. 날짜와 과제 데이터는 모두 **가상의 수업용 예시**입니다.

**실행 방법:** Colab에 노트북을 업로드한 후 `런타임 → 모두 실행`을 선택합니다. 추가 패키지나 API 키는 필요하지 않습니다.

## 1. 준비: 가상 과제 일정 JSON 만들기

실제 대학 LMS에 연결하지 않고, 학생들이 구조를 이해할 수 있도록 노트북 실행 시 연습용 JSON 파일을 생성합니다.

In [6]:
# json: 과제 데이터를 JSON 형식으로 저장하고 읽을 때 사용합니다.
import json
# date: 날짜를 처리하고 남은 날짜를 계산할 때 사용합니다.
from datetime import date
# Path: 파일 경로를 운영체제와 관계없이 다루는 도구입니다.
from pathlib import Path

# 실습 재현을 위해 기준 날짜를 고정합니다. 실제 오늘 날짜로 바꾸려면 date.today()를 사용하세요.
TODAY = date(2026, 11, 18)
# Colab과 일반 Jupyter에서 모두 사용할 수 있는 현재 폴더의 파일 경로입니다.
DATA_FILE = Path("assignments_sample.json")

# 모든 강의와 과제는 수업에서 사용하는 '가상 데이터'입니다.
sample_assignments = [
    {"course": "파이썬프로그래밍", "title": "미니 프로젝트", "deadline": "2026-11-25"},
    {"course": "자바프로그래밍", "title": "파일 입출력 과제", "deadline": "2026-11-22"},
    {"course": "지능시스템", "title": "ReAct 실습 보고서", "deadline": "2026-11-27"},
]

# ensure_ascii=False: 저장한 파일에 한글이 그대로 보이도록 설정합니다.
# indent=2: 사람이 읽기 편하도록 들여쓰기를 적용합니다.
DATA_FILE.write_text(json.dumps(sample_assignments, ensure_ascii=False, indent=2), encoding="utf-8")
print("연습용 과제 파일 생성:", DATA_FILE.resolve())
print("기준 날짜:", TODAY.isoformat(), "(가상 실습 기준일)")

연습용 과제 파일 생성: /content/assignments_sample.json
기준 날짜: 2026-11-18 (가상 실습 기준일)


## 2. 도구(Tool) 구현

- `get_assignment_deadline(course)`: 파일에서 강의의 과제와 마감일을 조회합니다.
- `get_current_date()`: 기준 날짜를 반환합니다.
- `days_until(deadline, today)`: 두 날짜 사이의 일수를 계산합니다.

**Observation은 도구가 실제로 반환한 값만 사용합니다.** 못 찾은 과제를 임의로 만들어 내지 않습니다.

In [2]:
# 도구 1: JSON 파일을 직접 읽어 수업별 과제를 찾습니다.
def get_assignment_deadline(course: str) -> dict:
    """수업명을 받아 과제 정보를 반환합니다. 없으면 오류를 나타냅니다."""
    # 파일에 저장된 최신 예제 데이터를 읽습니다. 모델이 과제를 추측하지 않습니다.
    assignments = json.loads(DATA_FILE.read_text(encoding="utf-8"))
    # 수업명 앞뒤 공백 때문에 조회가 실패하지 않게 정리합니다.
    search_name = course.strip()
    # 각 항목의 course 값이 요청한 수업명과 같은지 확인합니다.
    for assignment in assignments:
        if assignment["course"] == search_name:
            return {"ok": True, "assignment": assignment}
    # 조회 실패도 실제 도구 결과이며, 임의의 마감일을 반환하지 않습니다.
    return {"ok": False, "error": f"'{search_name}' 과제를 찾지 못했습니다."}

# 도구 2: 실습 기준일을 반환합니다.
def get_current_date() -> dict:
    """수업에서 지정한 기준 날짜를 반환합니다."""
    return {"ok": True, "today": TODAY.isoformat()}

# 도구 3: 입력한 날짜가 유효한지 검사한 뒤 남은 날짜를 계산합니다.
def days_until(deadline: str, today: str) -> dict:
    """마감일 - 기준일을 일(day) 단위 정수로 계산합니다."""
    try:
        # ISO 형식(YYYY-MM-DD) 문자열을 실제 날짜 객체로 바꿉니다.
        deadline_date = date.fromisoformat(deadline)
        today_date = date.fromisoformat(today)
        # 양수: 남음 / 0: 오늘 / 음수: 마감일이 지남
        remaining_days = (deadline_date - today_date).days
        return {"ok": True, "remaining_days": remaining_days}
    except ValueError:
        # 잘못된 날짜라면 실패 메시지를 반환하여 잘못된 계산을 막습니다.
        return {"ok": False, "error": "날짜 형식이 잘못되었습니다. YYYY-MM-DD를 사용하세요."}

# 각 도구가 정상 작동하는지 직접 확인합니다.
print("과제 조회:", get_assignment_deadline("파이썬프로그래밍"))
print("날짜 조회:", get_current_date())
print("일수 계산:", days_until("2026-11-25", "2026-11-18"))

과제 조회: {'ok': True, 'assignment': {'course': '파이썬프로그래밍', 'title': '미니 프로젝트', 'deadline': '2026-11-25'}}
날짜 조회: {'ok': True, 'today': '2026-11-18'}
일수 계산: {'ok': True, 'remaining_days': 7}


## 3. 간단한 질문 분석과 ReAct 루프

사용자 질문에 포함된 수업명을 찾아 조회합니다. 이 부분은 **LLM 추론이 아니라 명시적인 규칙**입니다. 이어서 `과제 조회 → 날짜 조회 → 남은 일수 계산`을 차례로 실행합니다.

실습에서의 용어:
- **Thought:** 지금 필요한 작업을 판단해 기록
- **Action:** 등록된 Python 도구 실행
- **Observation:** 그 도구가 반환한 **실제 값** 기록
- **Final Answer:** 필요한 정보가 충분하면 최종 답변 작성

In [7]:
# 질문에서 찾을 수업명 목록입니다. 예제 데이터와 일치해야 합니다.
KNOWN_COURSES = ["파이썬프로그래밍", "자바프로그래밍", "지능시스템"]

# 사용자 질문에 수업명이 나타나는지 간단한 문자열 검색으로 확인합니다.
def find_course(question: str):
    """질문에 포함된 수업명을 반환합니다. 없으면 None을 반환합니다."""
    for course in KNOWN_COURSES:
        if course in question:
            return course
    return None

# 도구의 이름을 실제 Python 함수에 연결해 둔 목록입니다.
TOOL_REGISTRY = {
    "get_assignment_deadline": get_assignment_deadline,
    "get_current_date": get_current_date,
    "days_until": days_until,
}

# 기록을 사람이 보기 편하게 단계별로 출력하는 함수입니다.
def print_trace(trace: list):
    """Thought / Action / Observation을 순서대로 출력합니다."""
    for step, item in enumerate(trace, start=1):
        print(f"\n[단계 {step}] Thought : {item['thought']}")
        print(f"         Action  : {item['action']}({item['arguments']})")
        print(f"         Observation: {item['observation']}")

# ReAct의 핵심인 도구 호출-관찰 반복을 구현합니다.
def react_assistant(question: str, max_steps: int = 3) -> dict:
    """질문을 받아 도구를 최대 3번 실행한 뒤 답변과 실행 기록을 반환합니다."""
    # 무한 반복을 막기 위해 최대 실행 횟수를 제한합니다.
    if max_steps < 1:
        raise ValueError("max_steps는 1 이상이어야 합니다.")
    # 과정 기록은 학생이 디버깅할 때 사용할 수 있습니다.
    trace = []
    # 수업명을 찾지 못하면 도구를 무작정 호출하지 않고 질문을 다시 요청합니다.
    course = find_course(question)
    if course is None:
        return {"answer": "수업명을 찾지 못했습니다. 파이썬프로그래밍, 자바프로그래밍, 지능시스템 중 하나를 입력하세요.", "trace": trace}

    # Observation에서 필요한 값을 따로 보관합니다.
    assignment = None
    today_text = None
    remaining_days = None

    # 이번 단계에서 어떤 도구를 호출할지 정해 둡니다.
    # 실제 LLM 기반 구현에서는 모델이 단계마다 이 결정을 내립니다.
    tasks = [
        ("과제 마감일을 확인해야 한다.", "get_assignment_deadline", lambda: {"course": course}),
        ("남은 날짜를 계산하려면 기준 날짜가 필요하다.", "get_current_date", lambda: {}),
        ("마감일과 기준 날짜를 이용해 남은 일수를 계산한다.", "days_until", lambda: {"deadline": assignment["deadline"], "today": today_text}),
    ]

    # 정해진 최대 횟수까지만 Thought → Action → Observation을 반복합니다.
    for thought, tool_name, make_arguments in tasks[:max_steps]:
        arguments = make_arguments()  # 앞 단계의 Observation이 다음 단계 입력이 됩니다.
        observation = TOOL_REGISTRY[tool_name](**arguments)  # 실제 도구를 호출합니다.
        trace.append({
            "thought": thought,
            "action": tool_name,
            "arguments": arguments,
            "observation": observation,
        })
        # 도구 오류가 발생하면 값을 지어내지 않고 바로 중단합니다.
        if not observation["ok"]:
            return {"answer": f"확인할 수 없습니다: {observation['error']}", "trace": trace}
        # 각 도구의 결과를 다음 단계에서 쓸 수 있도록 저장합니다.
        if tool_name == "get_assignment_deadline":
            assignment = observation["assignment"]
        elif tool_name == "get_current_date":
            today_text = observation["today"]
        elif tool_name == "days_until":
            remaining_days = observation["remaining_days"]

    # 도구 호출 제한 때문에 필요한 값이 부족하다면 완료한 척하지 않습니다.
    if remaining_days is None:
        return {"answer": "도구 호출 한도에 도달했습니다. 답변을 완료하려면 추가 확인이 필요합니다.", "trace": trace}

    # 남은 날짜가 양수인지, 0인지, 음수인지에 따라 문장을 바꿉니다.
    if remaining_days > 0:
        timing = f"{remaining_days}일 남았습니다"
    elif remaining_days == 0:
        timing = "오늘이 마감일입니다"
    else:
        timing = f"마감일이 {abs(remaining_days)}일 지났습니다"

    # 최종 답변은 도구의 반환값만 이용해서 작성합니다.
    answer = (f"[가상 실습 데이터] {course}의 '{assignment['title']}' 마감일은 "
              f"{assignment['deadline']}이며, {today_text} 기준 {timing}.")
    return {"answer": answer, "trace": trace}

## 4. 실행: 직접 질문해 보기

첫 질문에서는 세 도구 호출 기록과 최종 답변이 순서대로 나타납니다.

In [4]:
# 첫 번째 예제 질문입니다. 수업명을 바꿔도 됩니다.
question = "이번 학기 파이썬프로그래밍 과제 제출 마감일이 언제야? 며칠 남았어?"
# ReAct 시뮬레이터를 실행합니다.
result = react_assistant(question)
# 각 도구의 실제 결과를 먼저 보여줍니다.
print_trace(result["trace"])
# 최종 답변은 결과를 모두 확인한 뒤 표시합니다.
print("\n[Final Answer]", result["answer"])


[단계 1] Thought : 과제 마감일을 확인해야 한다.
         Action  : get_assignment_deadline({'course': '파이썬프로그래밍'})
         Observation: {'ok': True, 'assignment': {'course': '파이썬프로그래밍', 'title': '미니 프로젝트', 'deadline': '2026-11-25'}}

[단계 2] Thought : 남은 날짜를 계산하려면 기준 날짜가 필요하다.
         Action  : get_current_date({})
         Observation: {'ok': True, 'today': '2026-11-18'}

[단계 3] Thought : 마감일과 기준 날짜를 이용해 남은 일수를 계산한다.
         Action  : days_until({'deadline': '2026-11-25', 'today': '2026-11-18'})
         Observation: {'ok': True, 'remaining_days': 7}

[Final Answer] [가상 실습 데이터] 파이썬프로그래밍의 '미니 프로젝트' 마감일은 2026-11-25이며, 2026-11-18 기준 7일 남았습니다.


## 5. 확인: 예상 결과와 오류 처리

다른 과목, 존재하지 않는 과제, 도구 호출 제한을 차례로 검사합니다.

In [5]:
# 실행 결과가 예상과 일치하는지 assert로 검사합니다.
# 2026-11-18에서 2026-11-25까지 남은 날짜는 7일입니다.
assert "7일 남았습니다" in result["answer"]
assert len(result["trace"]) == 3  # 도구 3개가 순서대로 실행돼야 합니다.
assert result["trace"][0]["action"] == "get_assignment_deadline"
assert result["trace"][1]["action"] == "get_current_date"
assert result["trace"][2]["action"] == "days_until"

# 다른 강의에 대해서도 날짜를 올바르게 계산하는지 확인합니다.
java_result = react_assistant("자바프로그래밍 과제 언제까지야?")
assert "4일 남았습니다" in java_result["answer"]

# 질문에 수업명이 없으면 답을 추측하지 않고 추가 정보를 요청해야 합니다.
unknown_result = react_assistant("과제 마감일 알려줘")
assert "수업명을 찾지 못했습니다" in unknown_result["answer"]

# 파일에 없는 과제를 조회할 경우 실패 Observation이 돌아오는지 확인합니다.
missing_result = get_assignment_deadline("없는과목")
assert missing_result["ok"] is False

# 최대 2회 호출로 제한하면 계산이 끝나기 전에 정지해야 합니다.
limited_result = react_assistant("지능시스템 과제 알려줘", max_steps=2)
assert "도구 호출 한도" in limited_result["answer"]

print("모든 확인 테스트 통과!")
print("자바 질문:", java_result["answer"])
print("수업명 없는 질문:", unknown_result["answer"])
print("호출 제한 결과:", limited_result["answer"])

모든 확인 테스트 통과!
자바 질문: [가상 실습 데이터] 자바프로그래밍의 '파일 입출력 과제' 마감일은 2026-11-22이며, 2026-11-18 기준 4일 남았습니다.
수업명 없는 질문: 수업명을 찾지 못했습니다. 파이썬프로그래밍, 자바프로그래밍, 지능시스템 중 하나를 입력하세요.
호출 제한 결과: 도구 호출 한도에 도달했습니다. 답변을 완료하려면 추가 확인이 필요합니다.
